# Visual Question Answering with BLIP Fine-Tuning

This notebook demonstrates fine-tuning BLIP/CLIP models on a Visual Question Answering (VQA) dataset.

## Table of Contents
1. [Setup and Installation](#setup)
2. [Dataset Download and Exploration](#dataset)
3. [Model Initialization](#model)
4. [Data Preprocessing](#preprocessing)
5. [Training](#training)
6. [Evaluation](#evaluation)
7. [Inference](#inference)
8. [Visualization](#visualization)

## 1. Setup and Installation <a name="setup"></a>

In [ ]:
import os
import sys
from pathlib import Path

# Set cache directories to external drive to avoid disk space issues
os.environ['TRANSFORMERS_CACHE'] = '/media/nekoshou/New Volume1/VQA/.cache/transformers'
os.environ['HF_HOME'] = '/media/nekoshou/New Volume1/VQA/.cache/huggingface'
os.environ['TORCH_HOME'] = '/media/nekoshou/New Volume1/VQA/.cache/torch'

# Create cache directories
for cache_dir in [os.environ['TRANSFORMERS_CACHE'], os.environ['HF_HOME'], os.environ['TORCH_HOME']]:
    os.makedirs(cache_dir, exist_ok=True)

print("Environment setup complete!")
print(f"Transformers cache: {os.environ['TRANSFORMERS_CACHE']}")
print(f"HuggingFace cache: {os.environ['HF_HOME']}")

In [ ]:
# Import required libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import json

# Add src to path
sys.path.append(str(Path.cwd() / "src"))

from model_strategy import ModelFactory, BLIPStrategy
from dataset_processor import DatasetProcessor
from vqa_manager import VQAManager
from evaluator import VQAEvaluator
from database import VQADatabase

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Dataset Download and Exploration <a name="dataset"></a>

In [ ]:
# Download dataset using kagglehub
import kagglehub

# Set cache directory
cache_dir = "/media/nekoshou/New Volume1/VQA/.cache/kagglehub"
os.makedirs(cache_dir, exist_ok=True)
os.environ['KAGGLE_DATA_DIR'] = cache_dir

print("Downloading VQA dataset...")
dataset_path = kagglehub.dataset_download(
    "bhavikardeshna/visual-question-answering-computer-vision-nlp"
)

print(f"\nDataset downloaded to: {dataset_path}")

# Save path for later use
with open("/media/nekoshou/New Volume1/VQA/dataset_path.txt", 'w') as f:
    f.write(dataset_path)

In [ ]:
# Explore dataset structure
dataset_path = Path(dataset_path)
print("Dataset structure:")
for item in sorted(dataset_path.rglob("*"))[:20]:
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  {item.relative_to(dataset_path)}: {size_mb:.2f} MB")

In [ ]:
# Load and analyze dataset
processor = DatasetProcessor(str(dataset_path))
stats = processor.load_dataset()

print("\nDataset Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

# Analyze dataset characteristics
analysis = processor.analyze_dataset()
print("\nDataset Analysis:")
for key, value in analysis.items():
    if key != 'top_10_answers':
        print(f"  {key}: {value}")
    else:
        print(f"  {key}:")
        for ans, count in value:
            print(f"    {ans}: {count}")

## 3. Model Initialization <a name="model"></a>

In [ ]:
# Initialize BLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Create model strategy
model_strategy = ModelFactory.create_model("blip")
model_name = "Salesforce/blip-vqa-base"

print(f"Loading model: {model_name}")
model_strategy.load_model(model_name, device)

print(f"\nModel loaded successfully!")
print(f"Model type: {model_strategy.get_model_name()}")
print(f"Model parameters: {sum(p.numel() for p in model_strategy.model.parameters()):,}")

## 4. Data Preprocessing <a name="preprocessing"></a>

In [ ]:
# Determine image root directory
# This depends on your dataset structure
image_root = str(dataset_path)  # Adjust as needed

# Create dataloaders
batch_size = 8
num_workers = 4

print(f"Creating dataloaders with batch_size={batch_size}...")
train_loader, val_loader, test_loader = processor.create_dataloaders(
    processor=model_strategy.processor,
    image_root=image_root,
    batch_size=batch_size,
    num_workers=num_workers
)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader) if train_loader else 0}")
print(f"  Val batches: {len(val_loader) if val_loader else 0}")
print(f"  Test batches: {len(test_loader) if test_loader else 0}")

In [ ]:
# Visualize sample data
if train_loader:
    sample_batch = next(iter(train_loader))
    
    print("Sample batch:")
    print(f"  Pixel values shape: {sample_batch['pixel_values'].shape}")
    print(f"  Input IDs shape: {sample_batch['input_ids'].shape}")
    if 'labels' in sample_batch:
        print(f"  Labels shape: {sample_batch['labels'].shape}")
    
    # Display some examples
    print("\nSample questions and answers:")
    for i in range(min(3, len(sample_batch['question_text']))):
        print(f"\n  {i+1}. Q: {sample_batch['question_text'][i]}")
        print(f"     A: {sample_batch['answer_text'][i]}")

## 5. Training <a name="training"></a>

In [ ]:
# Setup VQA Manager
output_dir = "/media/nekoshou/New Volume1/VQA/outputs"
vqa_manager = VQAManager(
    model_strategy=model_strategy,
    device=device,
    output_dir=output_dir
)

# Setup training
learning_rate = 5e-5
weight_decay = 0.01

vqa_manager.setup_training(
    learning_rate=learning_rate,
    weight_decay=weight_decay
)

print("Training setup complete!")
print(f"  Learning rate: {learning_rate}")
print(f"  Weight decay: {weight_decay}")
print(f"  Output directory: {output_dir}")

In [ ]:
# Train the model
num_epochs = 3
gradient_accumulation_steps = 1

hyperparameters = {
    'model_name': model_name,
    'batch_size': batch_size,
    'num_epochs': num_epochs,
    'learning_rate': learning_rate,
    'weight_decay': weight_decay,
    'gradient_accumulation_steps': gradient_accumulation_steps
}

print(f"Starting training for {num_epochs} epochs...\n")

history = vqa_manager.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=num_epochs,
    gradient_accumulation_steps=gradient_accumulation_steps,
    hyperparameters=hyperparameters
)

print("\nTraining complete!")

## 6. Evaluation <a name="evaluation"></a>

In [ ]:
# Evaluate on test set
if test_loader:
    print("Evaluating on test set...")
    test_metrics = vqa_manager.evaluate(test_loader, "test")
    
    vqa_manager.evaluator.print_metrics(test_metrics)
    
    # Save metrics
    with open(f"{output_dir}/test_metrics.json", 'w') as f:
        json.dump(test_metrics, f, indent=2)
else:
    print("No test set available, evaluating on validation set...")
    test_metrics = vqa_manager.evaluate(val_loader, "validation")
    vqa_manager.evaluator.print_metrics(test_metrics)

## 7. Inference <a name="inference"></a>

In [ ]:
# Generate predictions
predictions = vqa_manager.predict(
    test_loader if test_loader else val_loader,
    save_results=True
)

print(f"\nGenerated {len(predictions)} predictions")
print("\nSample predictions:")
for i, pred in enumerate(predictions[:5]):
    print(f"\n{i+1}.")
    print(f"  Question: {pred['question']}")
    print(f"  Predicted: {pred['predicted_answer']}")
    print(f"  Ground Truth: {pred['ground_truth']}")
    print(f"  Match: {'✓' if pred['predicted_answer'].lower().strip() == pred['ground_truth'].lower().strip() else '✗'}")

In [ ]:
# Test inference on a custom image (if available)
# Replace with your own image path
try:
    sample_pred = predictions[0]
    test_image_path = sample_pred['image_path']
    test_question = "What is in the image?"
    
    if Path(test_image_path).exists():
        image = Image.open(test_image_path).convert('RGB')
        
        # Display image
        plt.figure(figsize=(8, 6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(f"Question: {test_question}")
        plt.show()
        
        # Generate answer
        answer = model_strategy.generate_answer(image, test_question)
        print(f"\nPredicted Answer: {answer}")
except Exception as e:
    print(f"Could not test custom inference: {e}")

## 8. Visualization <a name="visualization"></a>

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax = axes[0]
epochs = range(1, len(history['train_loss']) + 1)
ax.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
if history.get('val_loss'):
    ax.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Metrics plot
ax = axes[1]
metric_names = ['Accuracy', 'F1', 'BLEU-4']
metric_values = [
    test_metrics['accuracy'],
    test_metrics['f1_token'],
    test_metrics['bleu-4']
]
bars = ax.bar(metric_names, metric_values, color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.7)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Test Set Metrics', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(f"{output_dir}/training_results.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize sample predictions with images
import random

samples = random.sample(predictions, min(6, len(predictions)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, sample in enumerate(samples):
    ax = axes[i]
    
    try:
        img = Image.open(sample['image_path'])
        ax.imshow(img)
    except:
        ax.text(0.5, 0.5, 'Image not available', ha='center', va='center')
    
    ax.axis('off')
    
    # Format text
    q = sample['question'][:40] + '...' if len(sample['question']) > 40 else sample['question']
    pred = sample['predicted_answer'][:20]
    gt = sample['ground_truth'][:20]
    
    title = f"Q: {q}\nPred: {pred}\nGT: {gt}"
    color = 'green' if pred.lower().strip() == gt.lower().strip() else 'red'
    ax.set_title(title, fontsize=9, color=color)

plt.tight_layout()
plt.savefig(f"{output_dir}/sample_predictions.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Get error analysis
evaluator = VQAEvaluator()
for pred in predictions:
    evaluator.add_batch([pred['predicted_answer']], [pred['ground_truth']])

error_analysis = evaluator.get_error_analysis(top_k=10)

print("Error Analysis:")
print(f"  Total errors: {error_analysis['num_errors']}")
print(f"  Error rate: {error_analysis['error_rate']:.2%}")
print(f"\nMost common errors:")
for error, count in error_analysis['common_errors'][:10]:
    print(f"  {error}: {count}")

## Summary

This notebook demonstrated:
1. Loading and preprocessing a VQA dataset
2. Fine-tuning a BLIP model for visual question answering
3. Evaluating the model using multiple metrics (Accuracy, F1, BLEU)
4. Generating predictions and visualizing results
5. Logging experiments to a database

All results are saved in the output directory for further analysis.